# Feature Circuits

Traces how specific feature directions propagate through the network, measures feature composition scores, and attributes feature effects to paths.

**References:**
- Elhage et al. (2022) "Toy Models of Superposition"
- Bricken et al. (2023) "Towards Monosemanticity"

## Why This Matters

Feature circuits trace how SAE features propagate and compose across layers — a feature-level analog of circuit analysis. Composition scores, path attribution, and interaction matrices reveal the computational graph in terms of interpretable features rather than opaque activations.

**Key references:**
- [Bricken et al. (2023) "Towards Monosemanticity"](https://transformer-circuits.pub/2023/monosemantic-features/index.html) — Sparse autoencoders find interpretable features
- [Conmy et al. (2023) "Towards Automated Circuit Discovery"](https://arxiv.org/abs/2304.14997) — Automated methods for circuit finding via edge pruning

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.feature_circuits import (
    feature_propagation_trace,
    feature_composition_scores,
    feature_path_attribution,
    feature_interaction_matrix,
    feature_logit_effect,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35])
rng = np.random.RandomState(42)
direction = rng.randn(32).astype(np.float32)
direction /= np.linalg.norm(direction)
print(f"Model: {cfg.n_layers} layers, d_model={cfg.d_model}")

In [ ]:
# 1. Feature propagation trace
trace = feature_propagation_trace(model, tokens, direction)
print(f"Peak layer: {trace['peak_layer']}")
print(f"Emergence layer: {trace['emergence_layer']}")
print(f"\nProjection trajectory:")
for l, p in enumerate(trace['projections']):
    bar = '#' * int(abs(p) * 50)
    print(f"  Layer {l}: {p:+.4f} {bar}")
print(f"\nPer-layer contributions:")
for l in range(cfg.n_layers):
    print(f"  Layer {l}: attn={trace['attn_contributions'][l]:+.4f}, mlp={trace['mlp_contributions'][l]:+.4f}")

In [ ]:
# 2. Feature composition scores
d2 = rng.randn(32).astype(np.float32)
d2 /= np.linalg.norm(d2)
comp = feature_composition_scores(model, direction, d2)
print(f"Max OV head: {comp['max_ov_head']}, total OV score: {comp['total_ov_score']:.4f}")
print(f"Max QK head: {comp['max_qk_head']}")
print(f"\nOV scores:")
for l in range(cfg.n_layers):
    scores = [f"{comp['ov_scores'][l,h]:+.4f}" for h in range(cfg.n_heads)]
    print(f"  Layer {l}: {', '.join(scores)}")

In [ ]:
# 3. Feature path attribution
def metric(logits): return float(logits[-1, 0])
attr = feature_path_attribution(model, tokens, direction, metric)
print(f"Baseline: {attr['baseline_metric']:.6f}")
print(f"Dominant path: {attr['dominant_path']}")
print(f"\nAttention attributions:")
for l in range(cfg.n_layers):
    scores = [f"{attr['attn_attributions'][l,h]:.4f}" for h in range(cfg.n_heads)]
    print(f"  Layer {l}: {', '.join(scores)}")
print(f"MLP attributions: {[f'{x:.4f}' for x in attr['mlp_attributions']]}")

In [ ]:
# 4. Feature interaction matrix
dirs = [rng.randn(32).astype(np.float32) for _ in range(4)]
dirs = [d / np.linalg.norm(d) for d in dirs]
inter = feature_interaction_matrix(model, dirs, tokens)
print(f"Most interacting pair: {inter['most_interacting_pair']}")
print(f"Mean interaction: {inter['mean_interaction']:.4f}")
print(f"\nOV interaction matrix:")
for i in range(4):
    row = [f"{inter['ov_interaction_matrix'][i,j]:+.3f}" for j in range(4)]
    print(f"  [{', '.join(row)}]")

In [ ]:
# 5. Feature logit effect
eff = feature_logit_effect(model, tokens, direction, top_k=5)
print(f"Top promoted tokens: {eff['top_promoted']}")
print(f"  scores: {eff['promotion_scores'].round(4)}")
print(f"Top demoted tokens: {eff['top_demoted']}")
print(f"  scores: {eff['demotion_scores'].round(4)}")